# ACTRIS Aerosol Typing

Which aerosol was in the air, and at which altitude, from the ACTRIS aerosol remote sensing
archive.

This notebook searches the aerosol lidar stations served by the ACTRIS data centre, takes one
measurement, and runs the three aerosol classification products on it: **HIRAC**, **MAC** and
**HETEAC-Flex**. It ends by putting the three side by side, which is where they can be seen to
agree or to disagree.

The optical profiles - extinction, backscatter, lidar ratio, depolarisation - are not downloaded
and not plotted here. For those, and for the full analysis of a measurement, use
`ACTRIS_aerosol_remote_sensing.ipynb` in this same folder.

The search still covers the optical products, because that is how the data centre lists the
measurements: a typing product is requested for a measurement, and the measurement comes from
there.

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* before running the notebook
> again and before saving it: widget state belongs to the kernel that created it, and leftovers
> make the controls stop responding. If they still do after clearing, restart the kernel, and
> reload the browser page if that is not enough.

### Import libraries

### Dependencies

This notebook needs a few third-party packages. In the D4Science VRE the shared workspace shares
files, not the Python environment: each user has their own container, so a notebook that runs for
one person can fail for another with `ModuleNotFoundError`.

The cell below installs only the packages that are actually missing, so it costs nothing on an
environment that already has them.

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

import os
import importlib.util
import subprocess
import sys

# module name -> pip package name
REQUIRED = {
    "numpy": "numpy",
    "requests": "requests",
    "xarray": "xarray",
    "netCDF4": "netCDF4",        # backend xarray uses to read the NetCDF products
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    # Hidden dependency: plotly needs nbformat to render figures through the mime-type
    # renderer, but does not declare it as a hard requirement. A full Jupyter environment
    # always has it, so the omission only shows up in a clean one.
    "nbformat": "nbformat",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print("--> All required packages are already installed.")
else:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    # sys.executable, not '!pip': the shell's pip may belong to a different Python than the
    # one running this kernel, which is a classic source of "I installed it but it isn't found".
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    if result.returncode == 0:
        print("--> Done. If an import still fails below, restart the kernel and run again.")
    else:
        print("--> Installation failed:")
        print(result.stderr[-1500:])
        if "externally-managed-environment" in result.stderr:
            print("\n--> This environment does not allow pip to install into it directly.")
            print("    Ask the VRE administrators to add the packages, or retry with:")
            print(f"    {sys.executable} -m pip install --user " + " ".join(missing))

In [ ]:
# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ACTRIS Aerosol Typing"
NOTEBOOK_VERSION = "1.0"
NOTEBOOK_DATE = "2026-10-07"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

from pathlib import Path
import shutil
import re
import glob
import os
import zipfile
import datetime

import numpy as np
import requests
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MaxNLocator, MultipleLocator

from ipywidgets import (IntProgress, Dropdown, Layout, Button, Output, SelectMultiple,
                        DatePicker, Text, HBox, VBox)
from IPython.display import display, HTML

# Bump this whenever the notebook is updated, so you can tell at a glance which version the kernel
# is actually running (a stale kernel keeps old function definitions even after the file is updated).


### Select station

Pick a **country** first, then a **station** from that country. Two extra groups sit at the top of the
first dropdown: **"All stations"** to browse the full list, and **"Mobile stations"** for the mobile
(non-fixed) platforms such as OCEANET and LACROS — these are kept out of the per-country lists, since
they aren't tied to a fixed site.

The map below shows the stations of the selected group, with the currently highlighted station in red —
it updates as you change the dropdowns. When you're happy with the choice, click **"Confirm selection"**
and check that the "Confirmed station: ..." message appears.

⚠️ Just picking a value in the dropdowns is not enough - you must click the confirm button, otherwise
the notebook falls back to the default example station (`POT` - Potenza, Italy), telling you so when it does.
Re-running this cell keeps an already-confirmed station.

In [ ]:
r = requests.get("https://api.actris-ares.eu/api/services/restapi/stations")
r.raise_for_status()
stations_list = r.json()

try:
    import plotly.graph_objects as go
    _HAS_PLOTLY = True
except ImportError:
    _HAS_PLOTLY = False

stations_by_id = {s.get("ID"): s for s in stations_list}

def _country_name(code):
    """Human-readable country name, taken from the 'City, Country' part of any Location field."""
    for s in stations_list:
        if s.get("countryCode") == code and not _is_mobile(s):
            loc = s.get("Location") or ""
            if "," in loc:
                return loc.rsplit(",", 1)[1].strip()
    return code

def _is_mobile(s):
    """Mobile (non-fixed) platforms are flagged by the word 'mobile' in their Location field."""
    return "mobile" in (s.get("Location") or "").lower()

def _group_label(code):
    if code == "ALL":
        return "all stations"
    if code == "MOBILE":
        return "mobile platforms"
    return _country_name(code)

def _stations_for(code):
    if code == "ALL":
        sel = list(stations_list)
    elif code == "MOBILE":
        sel = [s for s in stations_list if _is_mobile(s)]
    else:
        # Fixed stations only: mobile platforms live in their own "Mobile stations" group
        sel = [s for s in stations_list if s.get("countryCode") == code and not _is_mobile(s)]
    return sorted(
        (f"{s.get('ID')} - {s.get('Location')} - {s.get('Institute')}", s.get("ID")) for s in sel
    )

_fixed_countries = sorted({
    s.get("countryCode") for s in stations_list if s.get("countryCode") and not _is_mobile(s)
})
country_options = [("All stations", "ALL")]
if any(_is_mobile(s) for s in stations_list):
    country_options.append(("Mobile stations", "MOBILE"))
country_options += [(f"{c} - {_country_name(c)}", c) for c in _fixed_countries]

# Preserve an already-confirmed station if this cell gets re-run
if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial_station = SELECTED_STATION if SELECTED_STATION in stations_by_id else "POT"
if _initial_station not in stations_by_id:
    _initial_station = stations_list[0].get("ID")
_initial_country = ("MOBILE" if _is_mobile(stations_by_id[_initial_station])
                    else stations_by_id[_initial_station].get("countryCode"))

country_dropdown = Dropdown(
    options=country_options, value=_initial_country, description="Country / group:",
    style={"description_width": "initial"}, layout=Layout(width="400px"),
)
station_dropdown = Dropdown(
    options=_stations_for(_initial_country), value=_initial_station, description="Station:",
    style={"description_width": "initial"}, layout=Layout(width="900px"),
)
confirm_button = Button(description="Confirm selection", button_style="success", icon="check")
confirm_output = Output()
map_output = Output()

def _draw_map(*_):
    """Show the stations of the selected country, highlighting the currently selected one."""
    with map_output:
        map_output.clear_output(wait=True)
        if not _HAS_PLOTLY:
            print("(plotly not available - install it to see the map)")
            return
        code = country_dropdown.value
        _ids = {v for _, v in _stations_for(code)}
        shown = [s for s in stations_list if s.get("ID") in _ids]
        current = station_dropdown.value
        others = [s for s in shown if s.get("ID") != current]
        sel = [s for s in shown if s.get("ID") == current]

        fig = go.Figure()
        if others:
            fig.add_trace(go.Scattergeo(
                lon=[s.get("Longitude") for s in others],
                lat=[s.get("Latitude") for s in others],
                text=[f"{s.get('ID')} - {s.get('Location')}" for s in others],
                hoverinfo="text", mode="markers", name="Other stations",
                marker=dict(size=8, color="#3B4872", line=dict(width=1, color="white")),
            ))
        if sel:
            fig.add_trace(go.Scattergeo(
                lon=[s.get("Longitude") for s in sel],
                lat=[s.get("Latitude") for s in sel],
                text=[f"{s.get('ID')} - {s.get('Location')}<br>{s.get('Institute')}" for s in sel],
                hoverinfo="text", mode="markers+text", name="Selected",
                textposition="top center", texttemplate="%{text}",
                marker=dict(size=15, color="crimson", line=dict(width=1.5, color="white")),
            ))
        # Scattergeo uses built-in geometry (no external map tiles needed)
        fig.update_geos(fitbounds="locations", resolution=50, showcountries=True,
                        countrycolor="#888", showland=True, landcolor="#f2efe9",
                        showocean=True, oceancolor="#dbe9f2", showlakes=False)
        fig.update_layout(height=420, margin=dict(l=0, r=0, t=30, b=0),
                          showlegend=False,
                          title=f"ACTRIS aerosol remote sensing stations - "
                                f"{_group_label(code)}")
        fig.show()

def _on_country_change(change):
    """Repopulate the station list when the country changes."""
    opts = _stations_for(change["new"])
    station_dropdown.unobserve(_on_station_change, names="value")
    station_dropdown.options = opts
    if opts:
        station_dropdown.value = opts[0][1]
    station_dropdown.observe(_on_station_change, names="value")
    _draw_map()

def _on_station_change(change):
    _draw_map()

def _on_confirm_clicked(b):
    global SELECTED_STATION
    SELECTED_STATION = station_dropdown.value
    with confirm_output:
        confirm_output.clear_output()
        s = stations_by_id.get(SELECTED_STATION, {})
        print(f"Confirmed station: {SELECTED_STATION} ({s.get('Location', '')})")

country_dropdown.observe(_on_country_change, names="value")
station_dropdown.observe(_on_station_change, names="value")
confirm_button.on_click(_on_confirm_clicked)

display(VBox([country_dropdown, station_dropdown, confirm_button, confirm_output, map_output]))
_draw_map()

if SELECTED_STATION:
    with confirm_output:
        print(f"Confirmed station: {SELECTED_STATION} (already confirmed - you can move to the next cell)")


### Select the date and time range

Pick the start/end date and time below and click **"Confirm date range"**, so you don't have to edit
any date string by hand in the code.

The defaults are set to the working example case used throughout this notebook — **Potenza (POT),
5 June 2025, 19:00:00 → 23:59:00** — so if you confirm nothing at all, the notebook still runs on a
known-good case. As with the other selectors, re-running this cell keeps an already-confirmed range.

In [ ]:
# TimePicker exists in ipywidgets >= 8; fall back to plain text boxes on older versions
try:
    from ipywidgets import TimePicker
    _HAS_TIMEPICKER = True
except ImportError:
    _HAS_TIMEPICKER = False

# Defaults = the working example case (Potenza, 5 June 2025, 19:00 -> 23:59)
DEFAULT_FROM_DATE = "2025-06-05"
DEFAULT_FROM_TIME = "19:00:00"
DEFAULT_TO_DATE = "2025-06-05"
DEFAULT_TO_TIME = "23:59:00"

if "SELECTED_DATE_RANGE" not in globals():
    SELECTED_DATE_RANGE = None
_init = SELECTED_DATE_RANGE or (DEFAULT_FROM_DATE, DEFAULT_FROM_TIME, DEFAULT_TO_DATE, DEFAULT_TO_TIME)

def _mk_time_widget(desc, value_str):
    if _HAS_TIMEPICKER:
        return TimePicker(
            description=desc,
            value=datetime.datetime.strptime(value_str, "%H:%M:%S").time(),
            style={"description_width": "initial"},
            layout=Layout(width="260px"),
        )
    return Text(
        description=desc,
        value=value_str,
        placeholder="HH:MM:SS",
        style={"description_width": "initial"},
        layout=Layout(width="260px"),
    )

def _read_time(widget):
    """Return the widget's value as an 'HH:MM:SS' string, whichever widget type it is."""
    v = widget.value
    if isinstance(v, str):
        return datetime.datetime.strptime(v.strip(), "%H:%M:%S").strftime("%H:%M:%S")
    return v.strftime("%H:%M:%S")

from_date_picker = DatePicker(
    description="From date:",
    value=datetime.datetime.strptime(_init[0], "%Y-%m-%d").date(),
    style={"description_width": "initial"}, layout=Layout(width="260px"),
)
from_time_picker = _mk_time_widget("From time:", _init[1])
to_date_picker = DatePicker(
    description="To date:",
    value=datetime.datetime.strptime(_init[2], "%Y-%m-%d").date(),
    style={"description_width": "initial"}, layout=Layout(width="260px"),
)
to_time_picker = _mk_time_widget("To time:", _init[3])

confirm_dates_button = Button(description="Confirm date range", button_style="success", icon="check")
confirm_dates_output = Output()

def _on_confirm_dates(b):
    global SELECTED_DATE_RANGE
    with confirm_dates_output:
        confirm_dates_output.clear_output()
        try:
            fd = from_date_picker.value.strftime("%Y-%m-%d")
            td = to_date_picker.value.strftime("%Y-%m-%d")
            ft = _read_time(from_time_picker)
            tt = _read_time(to_time_picker)
        except (AttributeError, ValueError):
            print("Invalid date/time: please make sure both dates are set and times are in HH:MM:SS format.")
            return
        if f"{td} {tt}" < f"{fd} {ft}":
            print("The end of the range is earlier than its start - please fix it and confirm again.")
            return
        SELECTED_DATE_RANGE = (fd, ft, td, tt)
        print(f"Confirmed range: {fd} {ft}  ->  {td} {tt}")

confirm_dates_button.on_click(_on_confirm_dates)

display(VBox([HBox([from_date_picker, from_time_picker]),
              HBox([to_date_picker, to_time_picker]),
              confirm_dates_button, confirm_dates_output]))

with confirm_dates_output:
    if SELECTED_DATE_RANGE:
        print(f"Confirmed range: {SELECTED_DATE_RANGE[0]} {SELECTED_DATE_RANGE[1]}  ->  "
              f"{SELECTED_DATE_RANGE[2]} {SELECTED_DATE_RANGE[3]} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_FROM_DATE} {DEFAULT_FROM_TIME} -> "
              f"{DEFAULT_TO_DATE} {DEFAULT_TO_TIME} (example case).")


### Configuration

This cell just collects the choices confirmed in the selector cells above. If you skipped a selector
(or forgot to click its confirm button), it falls back to the working example case — **Potenza (POT),
5 June 2025, 19:00:00 → 23:59:00** — and prints a message saying so, so the notebook always runs on
something valid.

**Data folder management**: to avoid accumulating files across repeated runs, this notebook wipes the
whole `data/` folder at the start of each run (set `CLEAN_DATA_FOLDER = False` to disable this) and then
creates a fresh subfolder for this run only, named `data/{STATION}_{FROM}_{TO}/`. All the files this run
produces — downloaded NetCDF files, the MAC,HIRAC and HETEAC-Flex zip archives, extracted files, and the saved
plot — are kept together inside that subfolder.

In [ ]:
# Everything below falls back to the working example case (Potenza, 5 June 2025, 19:00 -> 23:59)
# if you didn't confirm a choice in the selector cells above.
DEFAULT_STATION = "POT"
DEFAULT_FROM_DATE = globals().get("DEFAULT_FROM_DATE", "2025-06-05")
DEFAULT_FROM_TIME = globals().get("DEFAULT_FROM_TIME", "19:00:00")
DEFAULT_TO_DATE = globals().get("DEFAULT_TO_DATE", "2025-06-05")
DEFAULT_TO_TIME = globals().get("DEFAULT_TO_TIME", "23:59:00")

SELECTED_STATION = globals().get("SELECTED_STATION")
SELECTED_DATE_RANGE = globals().get("SELECTED_DATE_RANGE")

if SELECTED_STATION:
    STATION = SELECTED_STATION
else:
    STATION = DEFAULT_STATION
    print(f"No station confirmed - defaulting to the example station: {STATION}")

if SELECTED_DATE_RANGE:
    FROM_DATE, FROM_TIME, TO_DATE, TO_TIME = SELECTED_DATE_RANGE
else:
    FROM_DATE, FROM_TIME = DEFAULT_FROM_DATE, DEFAULT_FROM_TIME
    TO_DATE, TO_TIME = DEFAULT_TO_DATE, DEFAULT_TO_TIME
    print(f"No date range confirmed - defaulting to the example range: "
          f"{FROM_DATE} {FROM_TIME} -> {TO_DATE} {TO_TIME}")

print(f"Search parameters: station={STATION}, {FROM_DATE} {FROM_TIME} -> {TO_DATE} {TO_TIME}")

# Wipe the whole "data" folder before this run, so it doesn't grow indefinitely across executions.
# Set to False if you'd rather keep results from previous runs.
CLEAN_DATA_FOLDER = True

DATA_ROOT = Path("data")

# Close any NetCDF datasets left open from a previous run in this kernel: on network filesystems
# (NFS), deleting a file that a process still has open leaves behind an undeletable ".nfs..." file
# and makes shutil.rmtree raise "Device or resource busy".
for _ds_name in ("ds_extUV", "ds_extVIS", "ds_bscUV", "ds_bscVIS", "ds_bscIR", "ds_HIRAC"):
    _ds_obj = globals().get(_ds_name)
    if _ds_obj is not None:
        try:
            _ds_obj.close()
        except Exception:
            pass

if CLEAN_DATA_FOLDER and DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
        print("Note: some leftover files in 'data' could not be removed (still in use elsewhere); "
              "continuing anyway - they won't interfere with this run's own subfolder.")

DATA_ROOT.mkdir(exist_ok=True)

# Everything produced by this run (downloaded files, extracted archives, plots) goes into its own
# subfolder named after station + time window, so different runs don't mix their files together.
run_id = f"{STATION}_{FROM_DATE.replace('-', '')}{FROM_TIME.replace(':', '')}_{TO_DATE.replace('-', '')}{TO_TIME.replace(':', '')}"
outdir = DATA_ROOT / run_id
import shutil

outdir.mkdir(parents=True, exist_ok=True)
print(f"Output folder for this run: {outdir}")


# Figures are shown inline and kept in memory. Nothing is written to disk while plotting:
# the export cell at the end of the notebook writes out whatever you decide to keep, so the
# choice does not have to be made before drawing anything.
DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig


def sci_xaxis(ax, nticks=4):
    """Scientific notation on the x axis, with few ticks.

    Extinction and backscatter values are of the order of 1e-5 and 1e-7: written out in full
    the tick labels are long enough to overlap each other. Scientific notation moves the
    exponent to a single multiplier next to the axis.
    """
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nticks))
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0), useMathText=True)


def fold_x_offsets(fig):
    """Move the x10^n multiplier from the corner of the axis into the axis label.

    Matplotlib places that multiplier just below the right end of the axis, where it sits on
    top of a long axis label. Folding it into the label keeps both readable. The figure has
    to be drawn first, because the multiplier does not exist until then.
    """
    fig.canvas.draw()
    for ax in fig.axes:
        offset = ax.xaxis.get_offset_text().get_text()
        if offset:
            ax.xaxis.offsetText.set_visible(False)
            ax.set_xlabel(f"{ax.get_xlabel()} {offset}")

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


### Search the ACTRIS Aerosol Remote Sensing (ARES) archive

The REST API of the ACTRIS data centre lists the **optical** products of the station and period
chosen above. That is the right search even here: the archive is organised by measurement, each
measurement is identified through its optical products, and the typing products are then requested
for that measurement.

For more information, see <https://api.actris-ares.eu/api/swagger-ui/>.

In [ ]:
try:
    r = requests.get(
        "https://api.actris-ares.eu/api/services/restapi/opticalproducts/",
        params={
            "fromDate": FROM_DATE,
            "fromDayTime": FROM_TIME,
            "toDate": TO_DATE,
            "toDayTime": TO_TIME,
            "stations": STATION,
        },
    )
    r.raise_for_status()
    metadata = r.json()
    print(f"Data retrieved successfully! {len(metadata)} optical products found.")

except requests.exceptions.HTTPError as err:
    metadata = []
    if err.response.status_code == 404:
        print("No data available for the selected station in the specified time period.")
    else:
        print(f"An HTTP error occurred: {err}")

except Exception as e:
    metadata = []
    print(f"An unexpected error occurred: {e}")


### Collect the measurement identifiers

For every product found we keep its ID, its filename and its measurement ID. The NetCDF files
themselves are not downloaded: this notebook plots no optical profile, and the typing products are
fetched separately further down.

In [ ]:
# This notebook plots no optical product, so the NetCDF files are not downloaded: only the
# identifiers are collected, because a typing product is requested for a given measurement.
ids_filenames = {}
ids_measurementids = {}
for pr in metadata:
    ids_filenames[pr.get("ID")] = pr.get("Filename")
    ids_measurementids[pr.get("ID")] = pr.get("Measurement Id")

print(f"{len(ids_filenames)} optical product(s) found for this station and period.")
print("They are not downloaded here - they are what the measurement list is built from.")

### Select the measurement to analyse

The search may return products belonging to several distinct measurements. Pick the one you want to
analyse from the dropdown below and click **"Confirm measurement"**.

Each entry shows the `measurement ID` followed by the optical products available for it, so you can
tell at a glance which measurement has the wavelengths you need. If you don't confirm anything, the
notebook simply uses the **first measurement in the list** and says so, rather than stopping with an
error. As with the other selectors, this cell is safe to re-run: an already-confirmed measurement is
preserved.

In [ ]:
FNAME_RE = re.compile(r"_([eb])(\d{3,4})_")

def parse_type_wavelength(filename):
    """Return (type, wavelength_nm) parsed from an EARLINET optical-product filename, or (None, None)."""
    m = FNAME_RE.search(filename)
    if not m:
        return None, None
    kind, wl = m.group(1), m.group(2)
    # Wavelength codes are zero-padded (0355, 0532, 1064), so a plain int() conversion is correct:
    # do NOT slice the last 3 digits, that would turn 1064 into 64.
    wl_nm = int(wl)
    if not 250 <= wl_nm <= 2100:
        print(f"WARNING: implausible wavelength {wl_nm} nm parsed from '{filename}'. "
              f"If you see values like 64 nm, the kernel is running an outdated version of this "
              f"notebook - restart the kernel and run all cells from the top.")
    return kind, wl_nm

def product_label(kind, wl):
    return f"{'ext' if kind == 'e' else 'bsc'}@{wl}nm"

# Group product IDs by measurement
measurements = {}
for pid, mid in ids_measurementids.items():
    measurements.setdefault(mid, []).append(pid)

if not measurements:
    raise RuntimeError("No measurements found for the selected station/date range. "
                        "Try widening the date range or changing the station code.")

# Build a descriptive label for each measurement: id + list of available products
measurement_options = []
for mid in sorted(measurements):
    prods = []
    for pid in measurements[mid]:
        kind, wl = parse_type_wavelength(ids_filenames[pid])
        if kind is not None:
            prods.append(product_label(kind, wl))
    prods_str = ", ".join(sorted(set(prods))) if prods else "no recognised products"
    measurement_options.append((f"{mid}  -  [{prods_str}]", mid))

print(f"Found {len(measurement_options)} distinct measurement(s).")

# Preserve an already-confirmed measurement if this cell gets re-run
if "SELECTED_MEASUREMENT_ID" not in globals():
    SELECTED_MEASUREMENT_ID = None
_valid_ids = [v for _, v in measurement_options]
_initial_mid = SELECTED_MEASUREMENT_ID if SELECTED_MEASUREMENT_ID in _valid_ids else _valid_ids[0]

measurement_dropdown = Dropdown(
    options=measurement_options,
    value=_initial_mid,
    description="Measurement:",
    style={"description_width": "initial"},
    layout=Layout(width="900px"),
)
confirm_meas_button = Button(description="Confirm measurement", button_style="success", icon="check")
confirm_meas_output = Output()

def _on_confirm_meas(b):
    global SELECTED_MEASUREMENT_ID
    SELECTED_MEASUREMENT_ID = measurement_dropdown.value
    with confirm_meas_output:
        confirm_meas_output.clear_output()
        print(f"Confirmed measurement: {SELECTED_MEASUREMENT_ID}")

confirm_meas_button.on_click(_on_confirm_meas)

display(measurement_dropdown, confirm_meas_button, confirm_meas_output)

with confirm_meas_output:
    if SELECTED_MEASUREMENT_ID:
        print(f"Confirmed measurement: {SELECTED_MEASUREMENT_ID} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to the first measurement in the list: {_valid_ids[0]}")


### Resolve the measurement to request the typing products for

Filenames are parsed with the standard EARLINET naming convention:

`EARLINET_AerRemSen_{station}_Lev01_{type}{wavelength}_..._v{version}_qc{qc}.nc`

where `type` is `e` (extinction) or `b` (backscatter) and `wavelength` is a 4-digit code
(e.g. `0355`, `0532`, `1064`), so nothing has to be typed by hand.

In [ ]:
# If no measurement was confirmed above, fall back to the first one in the list so the notebook
# keeps running instead of erroring out.
SELECTED_MEASUREMENT_ID = globals().get("SELECTED_MEASUREMENT_ID")
if SELECTED_MEASUREMENT_ID not in measurements:
    SELECTED_MEASUREMENT_ID = sorted(measurements)[0]
    print(f"No measurement confirmed - defaulting to the first one: {SELECTED_MEASUREMENT_ID}")

selected_measurement_id = SELECTED_MEASUREMENT_ID
print(f"Selected measurement Id: {selected_measurement_id}")

# Build a lookup: (type, wavelength_nm) -> filename, for the selected measurement
files_by_type_wl = {}
for pid in measurements[selected_measurement_id]:
    fname = ids_filenames[pid]
    kind, wl = parse_type_wavelength(fname)
    if kind is not None:
        files_by_type_wl[(kind, wl)] = fname

print("Available optical products for this measurement:")
for (kind, wl), fname in sorted(files_by_type_wl.items()):
    label = "extinction" if kind == "e" else "backscatter"
    print(f"  {label} @{wl} nm -> {fname}")


## Aerosol type identification

The ACTRIS/EARLINET REST API also provides access to aerosol typing products (HIRAC, MAC and HETEAC-Flex).
More info: <https://earlinet.eu/our-knowledge/access-to-data/>

We use the `measurement ID` of the measurement selected above (`selected_measurement_id`) — no
manual lookup or hardcoded ID is needed.

In [ ]:
def download_product(kind):
    """Download one typing product archive, returning its filename or None.

    The endpoint answers 200 with an error page or an empty body when a product does not
    exist for the measurement, so the reply is checked before it is written: otherwise the
    problem only surfaces several cells later as "BadZipFile: File is not a zip file".
    """
    path = f"{selected_measurement_id}_{kind}.zip"
    try:
        r = requests.get(
            "https://api.actris-ares.eu/api/services/restapi/products/downloads",
            params={"kind": kind, "measurementId": selected_measurement_id},
            timeout=120,
        )
    except Exception as exc:  # noqa: BLE001 - a network failure must be readable
        print(f"{kind}: download failed ({type(exc).__name__}: {exc})")
        return None

    if r.status_code != 200:
        print(f"{kind}: the server answered {r.status_code}, no archive downloaded.")
        return None
    if not r.content:
        print(f"{kind}: the server returned an empty response - "
              f"this product is probably not available for this measurement.")
        return None
    if r.content[:2] != b"PK":                    # every zip starts with these two bytes
        preview = r.content[:200].decode("utf-8", "replace").strip().replace("\n", " ")
        print(f"{kind}: the reply is not a zip archive - this product is probably not "
              f"available for this measurement.")
        print(f"   the server said: {preview[:160]}")
        return None

    (outdir / path).write_bytes(r.content)
    # MAC and HETEAC are layer products of a few tens of kB, so a fixed MB unit would show
    # every one of them as "0.0 MB".
    size = len(r.content)
    shown = f"{size / 1e6:.1f} MB" if size >= 1e6 else f"{size / 1e3:.0f} kB"
    print(f"{kind}: {shown} downloaded")
    return path


productKind = "MAC"
productKind2 = "HIRAC"
productKind3 = "HETEAC"

# Downloaded and reported in the order the sections below use them
file_path2 = download_product(productKind2)   # HIRAC
file_path = download_product(productKind)     # MAC
file_path3 = download_product(productKind3)   # HETEAC-Flex

_products = (("HIRAC", file_path2), ("MAC", file_path), ("HETEAC", file_path3))
_got = [k for k, p in _products if p]
_missing = [k for k, p in _products if not p]
print(f"\nAvailable for this measurement: {', '.join(_got) if _got else 'none'}")
if _missing:
    print(f"Not available: {', '.join(_missing)} - the corresponding sections below will say so "
          f"and be skipped. This is normal: not every measurement has every typing product.")

### Extract the HIRAC archive

In [ ]:
if not file_path2:
    raise RuntimeError(
        "No HIRAC archive for this measurement, so the aerosol typing plots below cannot be "
        "produced. The MAC and HETEAC-Flex sections further down may still work."
    )

with zipfile.ZipFile(outdir / file_path2, "r") as zip_ref:
    zip_ref.extractall(outdir)
    extracted_names = zip_ref.namelist()

print("Extracted files:", extracted_names)

### Automatically find the HIRAC NetCDF file

Instead of typing the HIRAC filename by hand, we search the `data` folder for the extracted
`.nc` file containing "HIRAC" in its name.

In [ ]:
hirac_candidates = sorted(glob.glob(str(outdir / "**" / "*HIRAC*.nc"), recursive=True))
if not hirac_candidates:
    raise FileNotFoundError("No HIRAC .nc file found after extracting the archive.")

filename_HIRAC = Path(hirac_candidates[0]).name
print(f"Using HIRAC file: {filename_HIRAC}")

# Station location label for the plot title, taken from the station list fetched earlier
location = next(
    (f"{s.get('Location')}" for s in stations_list if s.get("ID") == STATION),
    STATION,
)


### Plot: HIRAC aerosol classification

In [ ]:
# Classes and colours of the HIRAC scheme, defined here rather than inside the try block so
# that the side-by-side comparison further down draws the same legend.
HIRAC_CLASSES = ["Clean", "No\nType", "Small", "Large\nspherical", "Mixture", "Large\naspherical"]
HIRAC_COLORS = ("#c7d8e5", "#999999", "#ddcc77", "#56b1f7", "#b44c1c", "#7f2f00")

# The plot is wrapped so that a missing file or an unexpected variable is reported in words
# rather than only as a traceback: these products are not published for every measurement.
try:
    ds_HIRAC = xr.open_dataset(Path(hirac_candidates[0]))
    type_mask = ds_HIRAC.aerosol_classification.fillna(0)
    height_f = ds_HIRAC.altitude
    time_f = ds_HIRAC.time

    name_f = filename_HIRAC.replace(".nc", "")

    plt.rc("font", family="serif")
    plt.rcParams.update({"font.size": 18})

    # Define categories
    cmap = mcolors.ListedColormap(HIRAC_COLORS)
    bounds = [0, 1, 2, 3, 4, 5, 6]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig = plt.figure(figsize=(10.0, 5.0))
    ax = fig.add_axes([0.07, 0.13, 0.99, 0.68])

    xx_, yy_ = np.meshgrid(time_f, height_f / 1000.0)
    cs1 = ax.pcolormesh(xx_, yy_, type_mask.transpose(), cmap=cmap, norm=norm)

    ax.set_ylim(0, 12)
    ax.set(ylabel="Altitude [km asl]")
    ax.set(xlabel="Time UTC [HH:MM]")

    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    ax.yaxis.set_minor_locator(MultipleLocator(1))
    ax.tick_params(which="both", top=True, right=True, direction="in")
    ax.set_title(f"Aerosol classification: {location}\n{name_f}", fontsize=18)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    cbar = plt.colorbar(
        cs1, cax=cax,
        format=lambda x, _: {0.5: "Clean", 1.5: "No\nType", 2.5: "Small", 3.5: "Large\nspherical",
                              4.5: "Mixture", 5.5: "Large\naspherical"}[x],
    )
    cbar.set_ticks([0.5, 1.5, 2.5, 3.5, 4.5, 5.5])
    cbar.ax.set_yticklabels(
        ["Clean", "No\nType", "Small", "Large\nspherical", "Mixture", "Large\naspherical"],
        rotation="horizontal", fontsize=12, va="center",
    )

    register_figure(fig, name_f)

    plt.show()
except (IndexError, FileNotFoundError):
    print("No HIRAC file to plot: the archive was not downloaded, or held no NetCDF file.")
    print("See the download section above - HIRAC is not published for every measurement.")
except (KeyError, AttributeError) as exc:
    print(f"The HIRAC file does not contain what this plot expects ({exc}).")
    print("The product format may have changed; the variables found in the file are listed by:")
    print("    import xarray as xr; print(xr.open_dataset(hirac_candidates[0]))")
except Exception as exc:  # noqa: BLE001 - anything else, but say what happened
    import traceback
    print(f"The HIRAC plot could not be produced ({type(exc).__name__}: {exc}).")
    print("Full details below.")
    traceback.print_exc()

### HIRAC discussion

Layers of different kinds are observed, with the lofted layer (around 8 km) clearly related to the
fire plume (small particles).

## MAC aerosol typing: layer classification

The MAC archive was downloaded together with HIRAC earlier in the notebook, but until now only
HIRAC was used. MAC describes the atmosphere as a set of **discrete layers**, each with a base, a
top and an aerosol classification — a complementary view to the continuous HIRAC field: fewer
details, but an explicit answer to "which layers are there, and what are they made of".

Each layer is drawn as a rectangle spanning its base-to-top range, coloured by aerosol type.

In [ ]:
def extract_typing(archive_name, label, pattern, exclude=()):
    """Extract a typing archive and list its NetCDF files, or explain why there are none."""
    if not archive_name:
        print(f"No {label} archive for this measurement - nothing to plot in this section.")
        return []
    try:
        with zipfile.ZipFile(outdir / archive_name, "r") as zip_ref:
            zip_ref.extractall(outdir)
            print("Extracted:", zip_ref.namelist())
    except zipfile.BadZipFile:
        print(f"The {label} archive is not a readable zip file. It may have been truncated; "
              f"re-run the download cell above.")
        return []

    found = sorted(glob.glob(str(outdir / "**" / pattern), recursive=True))
    if not found and exclude:
        found = sorted(f for f in glob.glob(str(outdir / "**" / "*.nc"), recursive=True)
                       if not any(k in Path(f).name for k in exclude))
    if not found:
        present = [Path(f).name for f in glob.glob(str(outdir / "**" / "*.nc"), recursive=True)]
        print(f"No {label} NetCDF file after extracting the archive. Files present: {present}")
        return []

    print(f"\n{len(found)} {label} file(s):")
    for f in found:
        print("  ", Path(f).name)
    return found


mac_files = extract_typing(file_path, "MAC", "*MAC*.nc",
                           exclude=("HIRAC", "HETEAC", "AerRemSen"))

### Plotting functions

Adapted from `plot_mac_column.py`. The colour of each aerosol class can be changed in
`AEROSOL_TYPES` below.

In [ ]:
from datetime import datetime

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator

# Classification tables. MAC and HETEAC-Flex use different schemes and a different variable
# name, but the layer geometry is the same, so the plotting code below is shared.
MAC_TYPES = {
    0: ("Not classified", "#bababa"),
    1: ("Clean Continental", "#abd9e9"),
    2: ("Smoke", "#6a51a3"),
    3: ("Dust", "#d7301f"),
    4: ("Marine", "#4575b4"),
    5: ("Polluted Continental", "#cbc9e2"),
    6: ("Polluted Dust", "#fc8d59"),
    7: ("Mixed Dust", "#fdcc8a"),
}

HETEAC_TYPES = {
    1: ("CS", "#abd9e9"),
    2: ("FSA", "#6a51a3"),
    3: ("FSNA", "#9e9ac8"),
    4: ("CNS", "#d7301f"),
    5: ("CNS/CS", "#fc8d59"),
    6: ("CNS/FSA", "#fdcc8a"),
    7: ("CNS/FSNA", "#fef0d9"),
    8: ("FSA/FSNA", "#cbc9e2"),
    9: ("FSNA/CS", "#f2f0f7"),
    10: ("Not-typed", "#bababa"),
}

_KM_UNITS = {"km", "kilometre", "kilometres", "kilometer", "kilometers"}


def _parse_typing_time(value):
    """Parse '2024-08-12T00:00:00Z' as written in the global attributes."""
    return datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ")


def _height_to_metres_factor(var):
    """Factor converting this variable's values to metres.

    The original scripts assumed base/top were in metres, but the products store them in
    kilometres (`units = "km"`), which produced an axis labelled "m" with values around 3.
    The unit is read from the variable attributes; if it is absent, the magnitude decides.
    """
    unit = str(var.attrs.get("units", "")).strip().lower()
    if unit in _KM_UNITS:
        return 1000.0
    if unit in {"m", "metre", "metres", "meter", "meters"}:
        return 1.0
    values = var.values
    guessed_km = len(values) > 0 and float(max(values)) < 50
    print(f"NOTE: no usable 'units' attribute on '{var.name}' (found {unit!r}); "
          f"assuming {'kilometres' if guessed_km else 'metres'} from the value range.")
    return 1000.0 if guessed_km else 1.0


def load_typing_layers(nc_path, type_var):
    """One dict per detected layer of a typing file, heights normalised to metres.

    `type_var` is the classification variable: 'aerosol_classification' for MAC,
    'aerosol_type' for HETEAC-Flex.
    """
    with xr.open_dataset(nc_path) as ds:
        t0 = _parse_typing_time(ds.attrs["measurement_start_datetime"])
        t1 = _parse_typing_time(ds.attrs["measurement_stop_datetime"])
        if type_var not in ds.variables:
            raise KeyError(f"'{type_var}' not found in {Path(nc_path).name}. "
                           f"Variables present: {list(ds.variables)}")
        to_m = _height_to_metres_factor(ds["base"])
        n = ds.sizes.get("layers", 0)
        return [
            dict(t0=t0, t1=t1,
                 base=float(ds["base"].values[i]) * to_m,
                 top=float(ds["top"].values[i]) * to_m,
                 aerosol_type=int(ds[type_var].values[i]))
            for i in range(n)
        ]


def _style_axes(ax, date_axis=False):
    ax.tick_params(which="both", bottom=True, top=True, left=True, right=True,
                   width=1, direction="in")
    if not date_axis:
        # AutoMinorLocator on a date axis adds clutter and fights the date locator
        ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(linewidth=1, color="w")
    for side in ("top", "bottom", "left", "right"):
        ax.spines[side].set_linewidth(1)
    ax.set_facecolor("lightgray")


def _legend_for(layers, types, ax, **kw):
    present = sorted({l["aerosol_type"] for l in layers})
    handles = [
        Line2D([0], [0], marker="s", linestyle="", markersize=10,
               markerfacecolor=types.get(t, (f"type {t}", "black"))[1],
               markeredgecolor="k",
               label=types.get(t, (f"type {t}", "black"))[0])
        for t in present
    ]
    ax.legend(handles=handles, **kw)


def plot_typing_profile(nc_path, types, product, type_var, height_units="km",
                        column_width=0.8, ax=None):
    """One typing file as a single column of stacked, colour-coded layers."""
    layers = load_typing_layers(nc_path, type_var)
    if not layers:
        raise ValueError(f"No layers found in {Path(nc_path).name}")

    scale = 1.0 if height_units == "m" else 1000.0
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(4.2, 5.2))

    for lyr in layers:
        _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
        ax.add_patch(Rectangle((-column_width / 2, lyr["base"] / scale), column_width,
                               (lyr["top"] - lyr["base"]) / scale,
                               facecolor=colour, edgecolor="k", linewidth=0.4))

    # Limits come from the data. The original scripts forced ax.set_ylim(0, 10), which only
    # makes sense in kilometres and hides everything when the units are metres.
    ymax = max(l["top"] for l in layers) / scale
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(0, ymax * 1.15)
    ax.set_xticks([])

    t0, t1 = layers[0]["t0"], layers[0]["t1"]
    ax.set_ylabel(f"Height a.s.l. ({height_units})")
    ax.set_xlabel(t0.strftime("%Y-%m-%d %H:%M") + " - " + t1.strftime("%H:%M UTC"))
    where = location if location is not None else globals().get("location", "")
    ax.set_title(f"{product} classification\n{where}")
    _legend_for(layers, types, ax, prop={"size": 7})
    _style_axes(ax)
    fig.tight_layout()
    return fig, ax


def plot_typing_curtain(nc_paths, types, product, type_var, height_units="km",
                        mode="auto", column_width=0.8, gap_ratio_threshold=20, ax=None, location=None):
    """Several typing files side by side.

    mode "continuous"  - real time axis, for profiles close together in time
         "categorical" - one evenly spaced column per file, for sparse case studies
         "auto"        - categorical when the files are spread far apart in time
    """
    per_file = [(p, load_typing_layers(p, type_var)) for p in nc_paths]
    per_file = [pf for pf in per_file if pf[1]]
    if not per_file:
        raise ValueError("No layers found in the given files.")
    per_file.sort(key=lambda pf: pf[1][0]["t0"])

    all_layers = [l for _, layers in per_file for l in layers]
    scale = 1.0 if height_units == "m" else 1000.0

    if mode == "auto":
        durations = [(l["t1"] - l["t0"]).total_seconds() for l in all_layers]
        typical = max(sorted(durations)[len(durations) // 2], 60.0)
        span = (per_file[-1][1][0]["t0"] - per_file[0][1][0]["t0"]).total_seconds()
        mode = "categorical" if span > gap_ratio_threshold * typical else "continuous"
        print(f"mode='auto' resolved to '{mode}'")

    # An axis can be supplied, so that this product can be drawn next to the others in the
    # comparison figure instead of in a figure of its own.
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(9, 1.4 * len(per_file)), 5))
    else:
        fig = ax.figure

    if mode == "continuous":
        for lyr in all_layers:
            _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
            x0 = mdates.date2num(lyr["t0"])
            ax.add_patch(Rectangle((x0, lyr["base"] / scale),
                                   mdates.date2num(lyr["t1"]) - x0,
                                   (lyr["top"] - lyr["base"]) / scale,
                                   facecolor=colour, edgecolor="k", linewidth=0.4))
        tmin = min(mdates.date2num(l["t0"]) for l in all_layers)
        tmax = max(mdates.date2num(l["t1"]) for l in all_layers)
        pad = max((tmax - tmin) * 0.05, 1 / 24 / 60)
        ax.set_xlim(tmin - pad, tmax + pad)
        ax.xaxis_date()
        # A fixed formatter on a short interval produces many overlapping labels; letting the
        # locator choose the ticks and the formatter adapt to the span avoids that.
        locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.set_xlabel("Time (UTC)")
    else:
        for i, (_, layers) in enumerate(per_file):
            for lyr in layers:
                _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
                ax.add_patch(Rectangle((i - column_width / 2, lyr["base"] / scale),
                                       column_width,
                                       (lyr["top"] - lyr["base"]) / scale,
                                       facecolor=colour, edgecolor="k", linewidth=0.4))
        ax.set_xlim(-0.7, len(per_file) - 0.3)
        ax.set_xticks(range(len(per_file)))
        ax.set_xticklabels([l[0]["t0"].strftime("%Y-%m-%d\n%H:%M") for _, l in per_file])
        ax.set_xlabel("Measurement date/time (UTC) - columns not to time scale")

    ymax = max(l["top"] for l in all_layers) / scale
    ax.set_ylim(0, ymax * 1.15)
    ax.set_ylabel(f"Height a.s.l. ({height_units})")
    ax.set_title(f"{product} classification\n{location}")
    _legend_for(all_layers, types, ax, loc="upper left", bbox_to_anchor=(1.01, 1),
                borderaxespad=0, title="Aerosol type")
    _style_axes(ax, date_axis=(mode == "continuous"))
    fig.tight_layout()
    return fig, ax


def typing_plot_controls(files, types, product, type_var, prefix):
    """Show the layout/units/save controls for a typing product and draw the plot.

    Used for both MAC and HETEAC-Flex, so the two sections stay in step.
    """
    if not files:
        print(f"No {product} file for this measurement - nothing to plot here.")
        return

    several = len(files) > 1

    mode_dd = Dropdown(
        options=[("One plot per file", "per-file"),
                 ("All files, real time axis", "continuous"),
                 ("All files, one column each", "categorical"),
                 ("Choose automatically", "auto")],
        value="auto" if several else "per-file",
        description="Layout:", style={"description_width": "initial"},
        layout=Layout(width="420px"))
    # Heights are always shown in kilometres: that is the convention for these products,
    # and it is what the algorithm authors expect to see. The plotting functions still take a
    # height_units argument, so metres remain available when calling them directly.
    units = "km"
    btn = Button(description="Redraw", button_style="success", icon="refresh")
    out = Output()

    def draw(_=None):
        with out:
            out.clear_output(wait=True)
            mode = mode_dd.value
            try:
                # Saving follows the notebook-wide control in the "Saving the figures"
                # section, like every other figure here, rather than a separate switch.
                if mode == "per-file":
                    for path in files:
                        fig, _ax = plot_typing_profile(path, types, product, type_var,
                                                       height_units=units)
                        register_figure(fig, Path(path).stem)
                        plt.show()
                else:
                    fig, _ax = plot_typing_curtain(files, types, product, type_var,
                                                   height_units=units, mode=mode)
                    register_figure(fig, f"{prefix}_curtain")
                    plt.show()
            except (KeyError, ValueError) as exc:
                # xarray errors run to several lines of installation advice; the first line
                # is the part that says what actually went wrong.
                reason = str(exc).split("\n")[0][:160]
                print(f"The {product} file could not be read as expected: {reason}")
                print("The product format may have changed, or the file may be incomplete. "
                      "Re-running the download cell above is worth a try.")
            except FileNotFoundError:
                print(f"The {product} file is no longer on disk. The run folder is emptied at "
                      f"the start of each run - re-run the download and extraction cells above.")
            except Exception as exc:  # noqa: BLE001 - anything else, but say what happened
                import traceback
                print(f"The {product} plot could not be produced "
                      f"({type(exc).__name__}: {exc}). Full details below.")
                traceback.print_exc()

    # The dropdowns redraw on their own: changing a value and seeing nothing happen until a
    # button is pressed is a poor way to explore the data. The button stays as a refresh.
    mode_dd.observe(draw, names="value")
    btn.on_click(draw)

    controls = ([mode_dd] if several else []) + [btn, out]
    display(VBox(controls))
    if not several:
        print(f"One {product} file for this measurement: a single profile is drawn. The layout "
              f"options only apply when a measurement yields several files.")
    draw()

### Plot

Each measurement normally yields a single MAC file, so one profile is drawn: a column of layers,
coloured by aerosol type.

When a measurement yields several files, a **Layout** menu appears with three ways of arranging
them — one plot per file, all files on a real time axis, or one evenly spaced column per file —
plus an automatic choice based on how far apart the profiles are in time.

To keep any of these figures, use the [export cell](#export) at the end of the notebook.

In [ ]:
typing_plot_controls(mac_files, MAC_TYPES, "MAC", "aerosol_classification", prefix="MAC")

## HETEAC-Flex aerosol typing

HETEAC-Flex is a third typing product for the same measurement. Like MAC it describes the
atmosphere as discrete layers, but it uses a different classification scheme, built on combinations
of coarse/fine and spherical/non-spherical, absorbing/non-absorbing components. The meaning of each
code is given with the plot below.

The plotting controls are the same as for MAC.

In [ ]:
heteac_files = extract_typing(file_path3, "HETEAC-Flex", "*HETEAC*.nc")

### Plot

As for MAC, a single file gives one profile; the **Layout** menu appears only when a measurement
yields several. To keep the figure, use the [export cell](#export) at the end.

The legend uses the HETEAC-Flex codes:

| Code | Meaning | | Code | Meaning |
|---|---|---|---|---|
| CS | coarse spherical | | CNS/CS | coarse non-spherical / coarse spherical |
| FSA | fine spherical absorbing | | CNS/FSA | coarse non-spherical / fine spherical absorbing |
| FSNA | fine spherical non-absorbing | | CNS/FSNA | coarse non-spherical / fine spherical non-absorbing |
| CNS | coarse non-spherical | | FSA/FSNA, FSNA/CS | mixtures |
| Not-typed | no classification assigned | | | |

In [ ]:
typing_plot_controls(heteac_files, HETEAC_TYPES, "HETEAC-Flex", "aerosol_type", prefix="HETEAC")

## The three algorithms side by side

HIRAC, MAC and HETEAC-Flex classify the same measurement independently, and they do not always
agree. Reading them one after the other makes the differences hard to see, so this section puts
them on one figure, sharing the altitude axis.

They are not the same kind of product, and the figure keeps that visible rather than hiding it:
HIRAC resolves the classification in time and altitude, so it is drawn as a curtain, while MAC and
HETEAC-Flex describe discrete layers and are drawn as columns. Each keeps its own colour scheme,
because the classes are not the same either — a disagreement between two panels is a disagreement
between algorithms, not a difference in how they are drawn.

In [ ]:
def compare_typing(hirac_path=None, mac_paths=None, heteac_paths=None, top_km=None):
    """One figure with whichever of the three typing products is available."""
    panels = []
    if hirac_path:
        panels.append(("HIRAC", "grid", hirac_path))
    if mac_paths:
        panels.append(("MAC", "layers", mac_paths))
    if heteac_paths:
        panels.append(("HETEAC-Flex", "layers", heteac_paths))

    if not panels:
        print("None of the three typing products is available for this measurement.")
        return None

    missing = [name for name in ("HIRAC", "MAC", "HETEAC-Flex")
               if name not in [p[0] for p in panels]]
    if missing:
        print(f"Not available for this measurement: {', '.join(missing)}.")
        print("Not every product exists for every measurement - the rest are drawn below.")

    fig, axes = plt.subplots(1, len(panels), figsize=(6.0 * len(panels), 5.5), sharey=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, (label, kind, source) in zip(axes, panels):
        try:
            if kind == "grid":
                with xr.open_dataset(source) as ds:
                    classification = ds.aerosol_classification.fillna(0)
                    grid_x, grid_y = np.meshgrid(ds.time, ds.altitude / 1000.0)
                    cmap = mcolors.ListedColormap(HIRAC_COLORS)
                    norm = mcolors.BoundaryNorm(list(range(len(HIRAC_COLORS) + 1)), cmap.N)
                    mesh = ax.pcolormesh(grid_x, grid_y, classification.transpose().values,
                                         cmap=cmap, norm=norm)
                    cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.02)
                    cbar.set_ticks([i + 0.5 for i in range(len(HIRAC_CLASSES))])
                    cbar.ax.set_yticklabels(HIRAC_CLASSES, fontsize=7, va="center")
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
                ax.set_xlabel("Time UTC")
            else:
                types = MAC_TYPES if label == "MAC" else HETEAC_TYPES
                type_var = "aerosol_classification" if label == "MAC" else "aerosol_type"
                plot_typing_curtain(source, types, label, type_var, ax=ax)
        except Exception as exc:  # noqa: BLE001 - one unreadable product must not lose the others
            ax.text(0.5, 0.5, f"{label}\ncould not be read\n({type(exc).__name__})",
                    ha="center", va="center", transform=ax.transAxes, fontsize=9)

        ax.set_title(label)
        if top_km:
            ax.set_ylim(0, top_km)

    # Each panel is built by a function that labels its own axes, so the wording differs from
    # one to the next; with a shared vertical axis only the leftmost label is needed.
    for ax in axes:
        ax.set_ylabel("")
    axes[0].set_ylabel("Altitude [km a.s.l.]")

    # Three panels side by side leave each one about a third of the width, and the time
    # formatter these products use writes the date underneath as well: without thinning the
    # ticks and tilting them the labels run into each other.
    for ax in axes:
        # These are date axes: a generic numeric locator would relabel them 12, 24, 36 ...
        locator = mdates.AutoDateLocator(minticks=2, maxticks=4)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        for label in ax.get_xticklabels():
            label.set_rotation(30)
            label.set_horizontalalignment("right")
        # The date that ConciseDateFormatter writes in the corner repeats in every panel and
        # is already in the measurement identifier above.
        ax.xaxis.get_offset_text().set_visible(False)

    fig.suptitle(f"Aerosol typing - {STATION.upper()} - "
                 f"measurement {SELECTED_MEASUREMENT_ID}", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    register_figure(fig, f"typing_comparison_{SELECTED_MEASUREMENT_ID}")
    plt.show()
    return fig


# The file lists come from the sections above; whichever is missing is simply left out.
# The result is assigned rather than left as the last expression: plot_typing_curtain and the
# cell below both display it otherwise, and the figure appears twice.
_hirac_candidates = globals().get("hirac_candidates") or []
_comparison = compare_typing(
    hirac_path=str(_hirac_candidates[0]) if _hirac_candidates else None,
    mac_paths=globals().get("mac_files"),
    heteac_paths=globals().get("heteac_files"),
)

<a id="export"></a>
## Take the results with you

Everything this run downloaded or produced is in the folder shown at the top of the notebook
(`data/<station>_<from>_<to>/`), which is **wiped at the start of the next run**. If you want to keep
any of it, download it before running the notebook again.

The cell below packs that folder into a single zip and gives you a download link. Two choices:

* **Figures and tables only** — the PNG, CSV and TXT files. Small, and usually what you want.
* **Everything** — the downloaded NetCDF products and archives as well. Useful to keep the raw data
  of a case study, but considerably larger.

The zip is written next to the notebook, so it also stays available in the file browser.

The archive is written into an `exports/` folder next to the notebook. That folder is emptied at the start of each run as well, so download what you need before running the notebook again.

In [ ]:
# Everything this run downloaded or produced lives in `outdir`. JupyterLab can only download
# one file at a time, so the cell below packs the folder into a single zip next to the notebook
# and gives you a link to it.

import os
import zipfile

from IPython.display import HTML

FIGURE_SUFFIXES = (".png", ".pdf", ".svg", ".txt", ".csv")


def _folder_size(folder, suffixes=None):
    """Total size in bytes of the files that would go into the archive."""
    return sum(f.stat().st_size for f in Path(folder).rglob("*")
               if f.is_file() and (suffixes is None or f.suffix.lower() in suffixes))


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


bundle_dd = Dropdown(
    options=[("Figures and tables only (PNG, CSV, TXT)", "figures"),
             ("Everything, including the downloaded data", "all")],
    value="figures", description="Include:",
    style={"description_width": "initial"}, layout=Layout(width="480px"))
bundle_btn = Button(description="Create archive", button_style="success", icon="file-archive-o")
bundle_out = Output()


def _make_bundle(_=None):
    with bundle_out:
        bundle_out.clear_output()
        if not outdir.exists():
            print(f"Nothing to pack: {outdir} does not exist yet.")
            return

        figures_only = bundle_dd.value == "figures"
        suffixes = FIGURE_SUFFIXES if figures_only else None

        # Write out any figure that was drawn but not saved, so the archive is complete even
        # if saving was left off while plotting.
        written_now = []
        for stem, fig in DRAWN_FIGURES.items():
            path = outdir / f"{stem}.png"
            if not path.exists():
                fig.savefig(path, dpi=150, bbox_inches="tight")
                written_now.append(path.name)
        if written_now:
            print(f"Wrote {len(written_now)} figure(s) that had not been saved yet: "
                  f"{', '.join(written_now)}")
        files = [f for f in sorted(outdir.rglob("*"))
                 if f.is_file() and (suffixes is None or f.suffix.lower() in suffixes)]
        if not files:
            print("Nothing to pack: no figure has been drawn yet in this session, and the run "
                  "folder holds no PNG, CSV or TXT. Run the plot cells above first.")
            return

        # The archive is written next to the notebook, so the download link below resolves
        archive = EXPORT_DIR / f"{outdir.name}{'_figures' if figures_only else ''}.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f, f.relative_to(outdir.parent))

        print(f"{len(files)} file(s), {_human(_folder_size(outdir, suffixes))} "
              f"-> {_human(archive.stat().st_size)} compressed")
        for f in files[:10]:
            print(f"   {f.relative_to(outdir)}")
        if len(files) > 10:
            print(f"   ... and {len(files) - 10} more")
        # FileLink produces a link that JupyterLab tries to *open* in the editor, which fails
        # on a zip with "is not UTF-8 encoded". The /files/ endpoint with download=1 hands the
        # file to the browser instead. The path is relative to the Jupyter root, i.e. the home
        # directory.
        try:
            rel = archive.resolve().relative_to(Path.home())
        except ValueError:
            rel = archive.name
        # Under JupyterHub every URL is prefixed with /user/<name>/. Without that prefix the
        # link hits the hub instead of this server, which answers with an HTML error page -
        # the browser then saves that page as a .html file instead of the archive.
        _prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
        url = f"{_prefix}/files/{rel}?download=1"
        display(HTML(
            f'<p style="margin-top:8px">'
            f'<a href="{url}" download target="_blank" '
            f'style="padding:8px 14px;background:#1B4053;color:white;text-decoration:none;'
            f'border-radius:5px;display:inline-block">Download {archive.name}</a></p>'
            f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
            f'<code>{archive.name}</code> in the file browser on the left and choose '
            f'<i>Download</i>.</p>'))


bundle_dd.observe(lambda _c: bundle_out.clear_output(), names="value")
bundle_btn.on_click(_make_bundle)
display(VBox([bundle_dd, bundle_btn, bundle_out]))
print("Choose what to include, then press 'Create archive'.")